# Exercise 4.2 — The decision boundary (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day4/day4-02-decision-boundary.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

In Exercise 4.1 the map of the logistic regression's probability had a black
line across it: to one side the model says `yes`, to the other `no`. That line
is the **decision boundary**. In this exercise you work it out from the
coefficients, see why it is straight, and move it with the threshold.

We use the same people as Exercise 4.1: age, systolic blood
pressure, and whether they had a cardiovascular event in the next ten years.
Both measurements matter, so the boundary is tilted.

- **Part 1.** Find the boundary by hand from the coefficients, draw it, and
  see it as the line where a plane is cut, in 3D.
- **Part 2.** Change the threshold and see the line move: every threshold
  gives its own line.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text

`data/cvd_risk.csv` holds 500 adults aged 40 to 80 seen at a health check:
their age, their systolic blood pressure in mmHg, and whether they had a
cardiovascular event (a heart attack or a stroke) in the following ten years.
If you do not have it yet, [**download `cvd_risk.csv`**](data/cvd_risk.csv)
into your `data` folder.

In [ ]:
cvd = pd.read_csv("data/cvd_risk.csv")

train, test = train_test_split(cvd, test_size=0.2, random_state=42)

train["cvd"].value_counts()

### Part 1 — Where the class changes

Fit a logistic regression with both measurements, the same way as in
Exercise 4.1:

In [ ]:
log_model = LogisticRegression(C=np.inf)
log_model.fit(train[["age", "systolic_bp"]], train["cvd"])

b0 = log_model.intercept_[0]
b1, b2 = log_model.coef_[0]          # age, blood pressure
print("intercept:", b0)
print("weights (age, blood pressure):", b1, b2)

The model is a straight line on the **log-odds** scale. Log-odds are another
way of writing a probability $p$: $\log\big(p / (1 - p)\big)$. A probability of
0.5 has log-odds 0; higher probabilities have positive log-odds, lower ones
negative. The sigmoid turns log-odds back into a probability.

$$\text{log-odds} = \beta_0 + \beta_1 \times \text{age} + \beta_2 \times \text{blood pressure}$$

By default the model says `yes` when the probability is 0.5 or more. At 0.5
the probability is shared equally between the two classes: 0.5 for an event and
0.5 for no event. Above it `yes` is the more likely class, below it `no`. So
the class changes where the probability is 0.5, which is **log-odds of 0**. So the boundary is every
combination of age and blood pressure where

$$\beta_0 + \beta_1 \times \text{age} + \beta_2 \times \text{blood pressure} = 0$$

For a given blood pressure, solve for age:

$$\text{age} = \frac{-\beta_0 - \beta_2 \times \text{blood pressure}}{\beta_1}$$

#### Task 1a — The boundary at a blood pressure of 120 and of 160

Round the coefficients printed above to four significant figures and calculate,
by hand, the age at which the class changes for a blood pressure of 120 mmHg.
Then do the same for 160 mmHg.

> **Stuck?**
>
> In order: the intercept, the blood-pressure weight, and the age weight. The
> intercept is negative, so minus the intercept is a positive number.

> **Worked answer**

In [ ]:
boundary_120 = (-(-15.06) - 0.05366 * 120) / 0.1159
boundary_160 = (-(-15.06) - 0.05366 * 160) / 0.1159
print(boundary_120, boundary_160)

> With a blood pressure of 120 the model predicts an event from about age 74;
> with 160, from about age 56.

Check it with `.predict_proba()`: a person on the boundary should get a
probability of 0.5.

In [ ]:
log_model.predict_proba(pd.DataFrame({"age": [74.4], "systolic_bp": [120]}))

Every point on the boundary satisfies one linear equation in age and blood
pressure, so the boundary is a **straight line**. Here it is, drawn from the
coefficients, over the test patients:

In [ ]:
age_line = np.array([40, 80])
fig, ax = plt.subplots()
for cls, mk, colour in [("no", "o", "steelblue"), ("yes", "^", "darkorange")]:
    rows = test[test["cvd"] == cls]
    ax.scatter(rows["age"], rows["systolic_bp"], marker=mk, color=colour, s=20, label=cls)
# the boundary written as blood pressure = (-b0 - b1 * age) / b2
ax.plot(age_line, (-b0 - b1 * age_line) / b2, color="black", linewidth=2)
ax.scatter([74.4, 55.9], [120, 160], color="red", s=50, zorder=3)
ax.set_xlim(40, 80)
ax.set_ylim(80, 200)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Systolic blood pressure (mmHg)")
ax.legend(title="Event")
ax.set_title("The decision boundary at a threshold of 0.5")
plt.show()

Above and to the right of the line the model says `yes`; below and to the left,
`no`. The line is tilted because both measurements count: an older person
needs a lower blood pressure to cross it.

#### The same line in three dimensions

The log-odds, $\beta_0 + \beta_1 \times \text{age} + \beta_2 \times \text{blood pressure}$,
are a flat, tilted **plane** above the age–blood pressure floor, like the plane
in Exercise 3.1. The grey sheet is height 0. Where the plane passes through the
sheet is the decision boundary, the black line: where the plane is above the
sheet the model says `yes`, where it is below, `no`. The test patients are drawn
on the grey sheet. Turn the plot with the mouse and scroll to zoom.

In [ ]:
# This plot needs the plotly package: pip install plotly
import plotly.graph_objects as go

ages = np.arange(40, 81, 1)
bps = np.arange(80, 201, 2)
A, S = np.meshgrid(ages, bps)

# the log-odds at every grid point: one row per blood pressure, one column per age
logodds_matrix = b0 + b1 * A + b2 * S

# the boundary: the age where the log-odds are 0, for each blood pressure
line_age = (-b0 - b2 * bps) / b1
on_floor = (line_age >= 40) & (line_age <= 80)

no = test[test["cvd"] == "no"]
yes = test[test["cvd"] == "yes"]

fig_logodds = go.Figure()
fig_logodds.add_trace(go.Surface(x=ages, y=bps, z=logodds_matrix, opacity=0.75,
    colorscale=[[0, "steelblue"], [1, "darkorange"]], showscale=False, name="log-odds"))
fig_logodds.add_trace(go.Surface(x=ages, y=bps, z=0 * logodds_matrix, opacity=0.4,
    colorscale=[[0, "grey"], [1, "grey"]], showscale=False, name="height 0"))
fig_logodds.add_trace(go.Scatter3d(x=line_age[on_floor], y=bps[on_floor],
    z=np.zeros(on_floor.sum()), mode="lines", line={"color": "black", "width": 8},
    name="boundary"))
fig_logodds.add_trace(go.Scatter3d(x=no["age"], y=no["systolic_bp"], z=np.zeros(len(no)),
    mode="markers", marker={"size": 3, "color": "steelblue"}, name="test: no"))
fig_logodds.add_trace(go.Scatter3d(x=yes["age"], y=yes["systolic_bp"], z=np.zeros(len(yes)),
    mode="markers", marker={"size": 3, "color": "darkorange"}, name="test: yes"))
fig_logodds.update_layout(scene={"xaxis_title": "Age (years)",
                                 "yaxis_title": "Systolic BP (mmHg)",
                                 "zaxis_title": "Log-odds of an event"})
fig_logodds.show()

The sigmoid turns each log-odds into a probability, which bends the flat plane
into an S-shaped sheet between 0 and 1. A log-odds of 0 is a probability of
0.5, so the sheet crosses the grey level at 0.5 along exactly the same black
line. The test patients sit at 1 (an event) and at 0 (no event). Click
**level 0.3** and **level 0.7** in the legend to show two more levels; Part 2
is about them.

In [ ]:
# the probability at every grid point: the sigmoid of the log-odds
prob_matrix = 1 / (1 + np.exp(-logodds_matrix))

fig_prob = go.Figure()
fig_prob.add_trace(go.Surface(x=ages, y=bps, z=prob_matrix, opacity=0.75,
    colorscale=[[0, "steelblue"], [1, "darkorange"]], showscale=False, name="probability"))
for t in [0.3, 0.5, 0.7]:
    fig_prob.add_trace(go.Surface(x=ages, y=bps, z=0 * prob_matrix + t, opacity=0.4,
        colorscale=[[0, "grey"], [1, "grey"]], showscale=False, showlegend=True,
        name=f"level {t}", visible=True if t == 0.5 else "legendonly"))
fig_prob.add_trace(go.Scatter3d(x=line_age[on_floor], y=bps[on_floor],
    z=np.full(on_floor.sum(), 0.5), mode="lines", line={"color": "black", "width": 8},
    name="boundary at 0.5"))
fig_prob.add_trace(go.Scatter3d(x=no["age"], y=no["systolic_bp"], z=np.zeros(len(no)),
    mode="markers", marker={"size": 3, "color": "steelblue"}, name="test: no"))
fig_prob.add_trace(go.Scatter3d(x=yes["age"], y=yes["systolic_bp"], z=np.ones(len(yes)),
    mode="markers", marker={"size": 3, "color": "darkorange"}, name="test: yes"))
fig_prob.update_layout(scene={"xaxis_title": "Age (years)",
                              "yaxis_title": "Systolic BP (mmHg)",
                              "zaxis_title": "Probability of an event"})
fig_prob.show()

> **Think about it**
>
> 1. As far as the model is concerned, 10 mmHg more blood pressure is worth how
>    many years of age? Where do you see that in the plot?
> 2. Two people: one is 70 with a blood pressure of 128, the other 55 with 165.
>    Which side of the line is each on?
> 3. A model with three predictors (say age, blood pressure and cholesterol) also
>    has a boundary. What shape would it be?

> **Worked answer**

In [ ]:
log_model.predict_proba(pd.DataFrame({"age": [70, 55], "systolic_bp": [128, 165]}))

> 1. About 5 years: 10 × 0.0537 / 0.1159 = 4.6. Adding 10 mmHg raises the
>    log-odds as much as 5 extra years. That is the tilt of the line: going 10
>    mmHg up, the boundary moves 5 years younger (from 74 at 120 mmHg to 56 at
>    160 mmHg).
> 2. The 70-year-old is just below the line (probability 0.48, `no`); the
>    55-year-old is above it (0.54, `yes`). The younger person is called `yes`
>    because of the blood pressure. Neither measurement decides alone.
> 3. Still flat: one linear equation in three measurements is a flat plane in
>    three dimensions. A logistic regression always has a flat boundary in the
>    predictors you give it.

### Part 2 — The threshold moves the line

The cut-off of 0.5 is called the **decision threshold**. Nothing forces it to
be 0.5: we could call someone `yes` from a probability of 0.3, or only from
0.7. A threshold $t$ is a probability, so it has
a log-odds, $\log\big(t / (1 - t)\big)$, and the boundary is where the model's
log-odds equal it:

$$\beta_0 + \beta_1 \times \text{age} + \beta_2 \times \text{blood pressure} = \log\left(\frac{t}{1-t}\right)$$

Only the right-hand side changes, so the line keeps its tilt and moves
sideways. In the 3D plot of Part 1 this is cutting the S-shaped sheet at a
different height: the 0.3 level meets it further down and to the left, the 0.7
level further up and to the right.

#### Task 2a — The log-odds of three thresholds

> **Stuck?**
>
> Both blanks are the threshold column, `thresholds["t"]`.

> **Worked answer**

In [ ]:
thresholds = pd.DataFrame({"t": [0.3, 0.5, 0.7]})
thresholds["log_odds"] = np.log(thresholds["t"] / (1 - thresholds["t"]))
thresholds["age_at_bp_140"] = (thresholds["log_odds"] - b0 - b2 * 140) / b1

thresholds

> The log-odds are −0.847, 0 and 0.847. For a blood pressure of 140 the class
> changes at age 58, 65 and 72.

In [ ]:
fig, ax = plt.subplots()
for cls, mk in [("no", "o"), ("yes", "^")]:
    rows = test[test["cvd"] == cls]
    ax.scatter(rows["age"], rows["systolic_bp"], marker=mk, color="grey", s=20, label=cls)
for t, colour in [(0.3, "blue"), (0.5, "red"), (0.7, "purple")]:
    lo = np.log(t / (1 - t))
    ax.plot(age_line, (lo - b0 - b1 * age_line) / b2, color=colour,
            linewidth=2, label=f"threshold {t}")
ax.set_xlim(40, 80)
ax.set_ylim(80, 200)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Systolic blood pressure (mmHg)")
ax.legend(fontsize=8)
ax.set_title("Three thresholds, three parallel boundaries")
plt.show()

> **Key idea**
>
> We could have picked another threshold, and we would have had a different line
> separating our classes in 2D: one line for each threshold. The 0.5 line,
> where both classes are equally likely, is only the default.

Everyone above a line is predicted `yes` at that threshold. Count them:

In [ ]:
test_probs = log_model.predict_proba(test[["age", "systolic_bp"]])[:, 1]

print("t = 0.3:", (test_probs >= 0.3).sum())
print("t = 0.5:", (test_probs >= 0.5).sum())
print("t = 0.7:", (test_probs >= 0.7).sum())

> **Think about it**
>
> Lowering the threshold from 0.5 to 0.3 calls many more test patients `yes`.
> On the plot, which patients change side when the line moves from 0.5 to 0.3:
> triangles (they had an event), circles (they did not), or both?

> **Worked answer**
>
> 40 patients are called `yes` at 0.3, 27 at 0.5 and 14 at 0.7. Moving the line
> down (0.3) brings more of the triangles onto the `yes` side, but it brings over
> circles too. Moving it up (0.7) does the opposite: fewer circles called `yes`,
> but more triangles missed. The model, its coefficients and the tilt of the line
> are the same; only the decision changed. Why you might want to move it is the
> subject of Exercise 4.3.

> **Learning points**
>
> - The decision boundary is where the predicted probability equals the
>   threshold. For a logistic regression that is where the log-odds equal
>   $\log\big(t/(1-t)\big)$: 0 for a threshold of 0.5.
> - A logistic regression's boundary is straight (flat) in its predictors.
>   Changing the threshold moves it without changing its tilt.